# 1. Data collection and preprocessing
This repository detailed the list of processing and analysis how to retrieve a continuous diurnal LST during peak summer months with high spatial resolution.

| Data                 | Data source     | Access / Download source                                               | Preprocessing | Reference                              |
|----------------------|-----------------|------------------------------------------------------------------------|---------------|----------------------------------------|
| Daytime and Nighttime – Coarse     | MODIS           | APPEEARS - https://appeears.earthdatacloud.nasa.gov/                   | OSGeo4W Shell | [Modis_preprocessing](preprocessing/modis.txt)     |
| 		       | Himawari SST    | WinSCP - https://winscp.net/eng/download.php                           | OSGeo4W Shell | [Himawari_preprocessing](preprocessing/himawari.txt)  |
| Daytime – Fine       | Landsat 8 and 9 | Google Earth Engine - GEE code (Ermida et al., 2020) https://github.com/sofiaermida/Landsat_SMW_LST                  | GEE | [Landsat_preprocessing](preprocessing/landsat.txt)   |
| Nighttime – Fine   | ECOSTRESS       | APPEEARS - https://appeears.earthdatacloud.nasa.gov/                   | ArcGIS - Spatial analyst | Mosaic and Raster calculator to convert Kelvin into Celcius |

# 2. Data description

In [1]:
import pandas as pd
# load the list of images
image_list = pd.read_csv(r"C:\Users\percy\Documents\studies\visual_studio_code\PhD-chapter2-data-fusion-full-diurnal-lst\xlsx\list_of_images_csv.csv")

# Preview first 40 rows
display(image_list.head(40))

,spatial,temporal,use,date,view_time_UTC,view_time_AEDT,file_ID
0,Fine,Daytime,Reference,7/01/2023,7/01/2023 0:04,7/01/2023 11:04,LC80920862023007LGN00
1,Fine,Daytime,Validation,8/01/2021,8/01/2021 0:10,8/01/2021 11:10,LC80930862021008LGN00
2,Fine,Daytime,Validation,19/01/2022,19/01/2022 0:10,19/01/2022 11:10,LC90930862022019LGN01
3,Fine,Daytime,Validation,20/01/2022,20/01/2022 0:04,20/01/2022 11:04,LC80920872022020LGN00
4,Fine,Nighttime,Reference,25/12/2022,25/12/2022 12:50,25/12/2022 23:50,ECO_L2T_LSTE.002_LST_doy2022359124954_aid0001_55S
5,Fine,Nighttime,Reference,25/12/2022,25/12/2022 12:50,25/12/2022 23:50,ECO_L2T_LSTE.002_LST_doy2022359125046_aid0001_55S
6,Fine,Nighttime,Validation,12/12/2021,12/12/2021 11:22,12/12/2021 22:22,ECO_L2T_LSTE.002_LST_doy2021346112222_aid0001_55S
7,Fine,Nighttime,Validation,9/02/2023,9/02/2023 12:50,9/02/2023 23:50,ECO_L2T_LSTE.002_LST_doy2023040114723_aid0001_55S
8,Coarse,Daytime,Reference,7/01/2023,6/01/2023 22:48,7/01/2023 9:48,MOD11A1.061_LST_Day_1km_doy2023007000000_aid0001
9,Coarse,Daytime,Validation,8/01/2021,7/01/2021 23:54,8/01/2021 10:54,MOD11A1.061_LST_Day_1km_doy2021008000000_aid0001


# 2. Coarse images: Mosaicking, Reprojection, Resampling
The STARFM preprocessing steps are shown below

## 2.2 Daytime
The himawari image does not fully cover the Port Philip bay. Extrapolation using IDW was conducted. The original resolution was 2km. It was resampled to 1km to fit with the MODIS data. Then the SST in the study area is extracted using this file
C:\Users\percy\Documents\studies\visual_studio_code\PhD-chapter2-data-fusion-full-diurnal-lst\shapefile\ppb_square_b.7z

Extraction of the MODIS using a full rectangle with width and height values multiple by 1km (Unit of MODIS)
C:\Users\percy\Documents\studies\visual_studio_code\PhD-chapter2-data-fusion-full-diurnal-lst\shapefile\MODIS_fishnet_diss.7z

After alignment with the reference extract the output using
C:\Users\percy\Documents\studies\visual_studio_code\PhD-chapter2-data-fusion-full-diurnal-lst\shapefile\MODIS_extractByMask.7z

Resampling to 30 m (Reference fine = Landasat). 
Corresponding file to get the width and height (same extent as reference image), the following file was used
C:\Users\percy\Documents\studies\visual_studio_code\PhD-chapter2-data-fusion-full-diurnal-lst\shapefile\modis_30m_fish_diss_e.7z

## 2.3 Nightime
Resampling to 70 m (Reference fine = ECOSTRESS).
Corresponding file to get the width and height (same extent as reference image), the following file was used

The full preprocessing workflow is available at:
- [Daytime data processing](preprocessing/daytime_preprocessing.txt)
- [Nighttime data processing](preprocessing/nighttime_preprocessing.txt)

# 3. STARFM
To run STARFM, the following repository was used (Mileva et al. 2018)
https://github.com/nmileva/starfm4py?tab=readme-ov-file

[STARFM parameters](preprocessing/starfm_parameters.txt) 



## 3.1 Some scenes of observed and predicted LST

## 3.2. Comparison against air temperature at BOM stations
The LST data for the target dates were compared with air temperature.
### 3.1.1 LST extraction
First, the fused STARFM outputs were exported to the output folder using filenames such as `d_refer_20230107`, in which the date is encoded as an 8-character string (`YYYYMMDD`) and preceded by an additional 8 characters.

Secondly, for each raster file in the output folder, the LST is extracted using the PWRS and CWS shapefile 
...\shapefile\wn_bom_pt.7z
To do the extraction, the following code was used in ARCGIS [PRWS and CWS LST](preprocessing/pwrs_cws_lst_extraction.txt)

### 3.1.2 LST vs Air temperature


In [ ]:
# load the extracted LST data
day_lst = pd.read_csv(r"C:\Users\percy\Documents\studies\visual_studio_code\PhD-chapter2-data-fusion-full-diurnal-lst\output\csv\day_pwrs_cws_lst.csv")
night_lst = pd.read_csv(r"C:\Users\percy\Documents\studies\visual_studio_code\PhD-chapter2-data-fusion-full-diurnal-lst\output\csv\night_pwrs_cws_lst.csv")

# LST Convert the date column to datetime format
day_lst['date'] = pd.to_datetime(day_lst['date'], format='%d/%m/%Y')
night_lst['date'] = pd.to_datetime(night_lst['date'], format='%d/%m/%Y')

# Add the Time_AEDT column
day_lst['Time_AEDT'] = pd.to_datetime(day_lst['date']) + pd.Timedelta(hours=11)
night_lst['Time_AEDT'] = pd.to_datetime(night_lst['date']) + pd.Timedelta(hours=24)

# Merge day_lst with night_lst
day_night_lst = pd.concat([day_lst, night_lst], ignore_index=True)

# Preview first 40 rows
display(day_night_lst.head(5))

,p_id,source,date,lst_pred,Time_AEDT
0,1004,cws,2022-12-03,33.198058,2022-12-03 11:00:00
1,1010,cws,2022-12-03,37.639220,2022-12-03 11:00:00
2,1011,cws,2022-12-03,37.363447,2022-12-03 11:00:00
3,1016,cws,2022-12-03,37.205113,2022-12-03 11:00:00
4,1017,cws,2022-12-03,37.664265,2022-12-03 11:00:00


In [ ]:
# load the air temperature data with UHI inn Chapter 1
air_temp = pd.read_csv(r"C:\Users\percy\Documents\studies\visual_studio_code\chapter1_bis\output\uhii_delta_date_lcz_filtered.csv")

# Keep only relevant columns
air_temp = air_temp[['p_id', 'Time_AEDT', 'Time_UTC', 'ta']]

# Preview first 5 rows
display(air_temp.head(5))

c:\Users\percy\Documents\studies\visual_studio_code\PhD-chapter2-data-fusion-full-diurnal-lst\.venv\lib\site-packages\IPython\core\interactiveshell.py:3553: DtypeWarning: Columns (3) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


,p_id,Time_AEDT,Time_UTC,ta
0,1,2023-01-06 06:00:00,2023-01-05 19:00:00,10.28
1,1,2023-01-06 07:00:00,2023-01-05 20:00:00,11.83
2,1,2023-01-06 08:00:00,2023-01-05 21:00:00,16.28
3,1,2023-01-06 09:00:00,2023-01-05 22:00:00,18.67
4,1,2023-01-06 10:00:00,2023-01-05 23:00:00,20.78


In [25]:
# Make sure both Time_AEDT columns are datetime
air_temp['Time_AEDT'] = pd.to_datetime(air_temp['Time_AEDT'])
day_night_lst['Time_AEDT'] = pd.to_datetime(day_night_lst['Time_AEDT'])

# Merge LST with air temperature data
lst_air_temp = pd.merge(day_night_lst, air_temp, on=['p_id', 'Time_AEDT'], how='inner')

# Order columns
lst_air_temp = lst_air_temp[['p_id', 'source', 'date', 'Time_AEDT', 'Time_UTC', 'lst_pred', 'ta']]

# Preview first 5 rows
display(lst_air_temp.head(5))

,p_id,source,date,Time_AEDT,Time_UTC,lst_pred,ta
0,1144,cws,2022-12-03,2022-12-03 11:00:00,2022-12-03 00:00:00,38.067105,27.0
1,1182,cws,2022-12-03,2022-12-03 11:00:00,2022-12-03 00:00:00,36.068465,24.2
2,1185,cws,2022-12-03,2022-12-03 11:00:00,2022-12-03 00:00:00,38.500715,25.5
3,1197,cws,2022-12-03,2022-12-03 11:00:00,2022-12-03 00:00:00,38.755817,24.8
4,1203,cws,2022-12-03,2022-12-03 11:00:00,2022-12-03 00:00:00,39.121011,21.9


In [27]:
# Filter PRWS only
lst_air_temp_prws = lst_air_temp[lst_air_temp['source'] == 'prws']

# Preview first 5 rows
display(lst_air_temp_prws.head(10))

,p_id,source,date,Time_AEDT,Time_UTC,lst_pred,ta
118,400,prws,2022-12-03,2022-12-03 11:00:00,2022-12-03 00:00:00,33.751797,26.1
119,200,prws,2022-12-03,2022-12-03 11:00:00,2022-12-03 00:00:00,37.289041,26.7
120,600,prws,2022-12-03,2022-12-03 11:00:00,2022-12-03 00:00:00,37.697760,28.3
121,500,prws,2022-12-03,2022-12-03 11:00:00,2022-12-03 00:00:00,38.241858,27.3
241,400,prws,2022-12-27,2022-12-27 11:00:00,2022-12-27 00:00:00,36.009568,31.6
242,200,prws,2022-12-27,2022-12-27 11:00:00,2022-12-27 00:00:00,39.431096,32.6
243,600,prws,2022-12-27,2022-12-27 11:00:00,2022-12-27 00:00:00,41.721304,33.4
244,500,prws,2022-12-27,2022-12-27 11:00:00,2022-12-27 00:00:00,38.050427,31.5
368,400,prws,2023-01-27,2023-01-27 11:00:00,2023-01-27 00:00:00,33.098478,20.1
369,200,prws,2023-01-27,2023-01-27 11:00:00,2023-01-27 00:00:00,33.135591,20.2


## 3.3. Assessment of temporal stability of fused LST 

## 3.4. Assessment of cross-date spatial pattern consistency

### a. Hotspot analysis

### b. Directional ring profiling using the urban-rural LST gradients

# References
- Ermida, S. L., Soares, P., Mantas, V., Göttsche, F.-M., & Trigo, I. F. (2020). Google earth engine open-source code for land surface temperature estimation from the landsat series. Remote Sensing, 12(9), 1471. 
- Gao, F., Masek, J., Schwaller, M., & Hall, F. (2006). On the blending of the Landsat and MODIS surface reflectance: Predicting daily Landsat surface reflectance. IEEE Transactions on Geoscience and Remote sensing, 44(8), 2207-2218.
- Mileva, N., Mecklenburg, S., & Gascon, F. (2018). New tool for spatio-temporal image fusion in remote sensing: A case study approach using Sentinel-2 and Sentinel-3 data. Image and Signal Processing for Remote Sensing Xxiv
